# Session 3: Baselines
**Emission-trajectory ML project. Dr. Khawar Naeem, QTTSC, Qatar University.**

Two models that require no learning: persistence and the linear trend. They cost minutes and define the bar every later model must clear. Skipping them is the most common credibility failure in forecasting portfolios; a Session 5 claim that "XGBoost achieves MAE of X" means nothing until X stands next to what doing-nothing achieves.

**Session contract:** everything here is evaluated on the VALIDATION years (targets 2015-2018) only. The test years stay untouched until Session 5's single final evaluation. Deliverables: this notebook executed, `results/model_comparison.csv`, and your three check-question answers.

## 0. Pre-registration (fill this in BEFORE running any cell below)

Committing to a prediction before seeing the result is the cheapest scientific habit there is: it converts the output from "a number I look at" into "a test of whether I understand the system."

**My predictions (Khawar, write before running):**

1. Overall validation MAE: which baseline wins, persistence or linear trend? Why?
   > *your answer here*
2. In roughly what fraction of countries does trend beat persistence? (Recall figure 3: median year-over-year change is ~4.4% of level, but direction reverses often.)
   > *your answer here*
3. For Qatar specifically, 2015-2018: which baseline wins? (You know Qatar's trajectory from the anchor table.)
   > *your answer here*

## 1. Load the modeling table and take the validation slice

Both baselines read columns that `build_features.py` already computed, which is deliberate: the baselines and the ML models will consume the exact same table, so no model can win through a data advantage.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import evaluate  # our shared metrics: every model in this project is scored through this module

df = pd.read_csv("../data/processed/modeling_table.csv")
val = df[df["split"] == "val"].copy()
print(f"validation rows: {len(val)}, countries: {val['country'].nunique()}, target years {val['target_year'].min()}-{val['target_year'].max()}")

## 2. Persistence: next year equals this year

The prediction is literally the `co2` column: no fitting, no parameters. Its error IS the year-over-year change we measured in Session 1 (median ~4.4% of level), which is why it is strong: emissions systems have massive inertia (power plants, vehicle fleets, and industrial capital do not turn over in a year).

In [ ]:
val["pred_persistence"] = val["co2"]

## 3. Linear trend: extrapolate the recent direction

Prediction: this year's level plus the average yearly change over the last five years. Note that `co2 + co2_slope5` reuses the slope feature from Session 2; the feature was designed so that the trend baseline and the ML models share the same notion of "recent direction."

The trade embedded here: trend beats persistence when a country keeps moving the same way (Qatar-style steady growth), and loses when direction reverses, because it confidently extrapolates the past into a turn it cannot see.

In [ ]:
val["pred_trend"] = val["co2"] + val["co2_slope5"]

## 4. The first comparison table

`evaluate.comparison_table` enforces the same-rows rule (any row missing either prediction is dropped for both models) and computes the skill score against persistence. Before running: your pre-registered answer to question 1 is about to be tested.

In [ ]:
table = evaluate.comparison_table(
    val,
    {"persistence": "pred_persistence", "linear_trend": "pred_trend"},
    persistence_col="pred_persistence",
)
table.round(3)

**Read the table against your predictions.** Things to notice: the gap between MAE and MedianAE (that gap is the giants); whether RMSE ranks the two models the same way MAE does (if not, one model has a fatter tail of disasters); and the skill number, which for the trend model answers "does knowing the direction help, net, across all countries?".

## 5. Where does each baseline win?

The overall table hides the geography. Count, per country, which baseline has the lower absolute error across the four validation years, then look at the two error-by-country tables. Expect the worst-MAE list to be a list of the biggest emitters; check whether the percentage column tells a different story.

In [ ]:
val_scored = val.dropna(subset=["target_co2_next", "pred_persistence", "pred_trend"]).copy()
val_scored["ae_persistence"] = (val_scored["target_co2_next"] - val_scored["pred_persistence"]).abs()
val_scored["ae_trend"] = (val_scored["target_co2_next"] - val_scored["pred_trend"]).abs()

per_country = val_scored.groupby("country")[["ae_persistence", "ae_trend"]].mean()
trend_wins = (per_country["ae_trend"] < per_country["ae_persistence"]).mean()
print(f"trend beats persistence in {trend_wins:.0%} of countries  (your pre-registered guess: ?)")

In [ ]:
evaluate.error_by_country(val_scored, "pred_persistence").head(10)  # worst absolute misses: who are they?

In [ ]:
# Qatar: your third pre-registered prediction
per_country.loc[["Qatar", "China", "United States", "Germany"]].round(3)

## 6. Save the results artifact

The results file, not the notebook output, is the project's memory. Session 4 will append Ridge and the tree model to this same file, and the README will cite it. Convention: one row per (model, split), overwritten only by a full rerun.

In [ ]:
table.insert(1, "split", "val")
table.to_csv("../results/model_comparison.csv", index=False)
print("saved -> results/model_comparison.csv")
table.round(3)

## 7. Reconcile and interpret

Write four sentences, in your own words, in this cell (double-click to edit):

1. Which pre-registrations were right, which wrong, and what did the wrong ones misunderstand?
   > *...*
2. What does the MAE-versus-MedianAE gap say about who dominates the average?
   > *...*
3. One sentence you could put in the README today, tied to `results/model_comparison.csv`.
   > *...*
4. What must Ridge (Session 4) beat, exactly: which number in which file?
   > *...*

## Check questions (close Session 3 by answering these to Claude)

1. For Qatar 2019-2024, is persistence biased up or down, and why? Would the same hold for a country whose emissions peaked in 2015?
2. Construct (or describe) a small example where MAE and RMSE rank two models differently. What kind of error pattern causes the flip?
3. A model posts skill of -0.05 on validation. Say precisely what that means in words, and name one legitimate reason you might still keep studying that model rather than discarding it.

**End-of-session protocol:** commit this notebook, `src/evaluate.py`, and `results/model_comparison.csv`; update CLAUDE.md status and session log; record the exact next action (Session 4: preprocessing pipeline fitted on train only, Ridge, and the tree benchmark).